# Task 1 - Day 1: Data Cleaning & Preprocessing

### Objective:
Learn how to clean and prepare raw data for ML models.

### Tools:
Python, Pandas, NumPy, Matplotlib, Seaborn

## Step 1: Imports and Loading the Dataset
We import standard packages and load the Titanic dataset from Seaborn (with a fallback URL to GitHub if Seaborn cannot download it).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create directory for plots
os.makedirs('plots', exist_ok=True)

try:
    df = sns.load_dataset('titanic')
    print("Loaded Titanic dataset successfully from Seaborn.")
except Exception as e:
    print(f"Failed to load via Seaborn ({e}). Attempting download from GitHub...")
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    print("Loaded Titanic dataset from GitHub successfully.")

## Step 2: Exploratory Data Analysis (EDA)
We investigate the dataset's basic info, shapes, data types, and check for missing values.

In [ ]:
print(f"Dataset Shape: {df.shape}\n")
print("Columns and Data Types:")
print(df.dtypes)

print("\nMissing Values Count per Column:")
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])

## Step 3: Handling Missing Values
- Drop column `deck` since it contains >70% nulls.
- Impute missing values for `age` using its median.
- Impute missing values for `embarked` and `embark_town` using their modes.
- Drop redundant/collinear columns (`alive`, `alone`, `class`, `who`, `adult_male`, `embark_town`).

In [ ]:
# Drop deck column
if 'deck' in df.columns:
    print("Dropping 'deck' column.")
    df.drop(columns=['deck'], inplace=True)

# Impute age with median
if 'age' in df.columns:
    age_median = df['age'].median()
    print(f"Imputing age with median: {age_median}")
    df['age'].fillna(age_median, inplace=True)
elif 'Age' in df.columns:
    age_median = df['Age'].median()
    print(f"Imputing Age with median: {age_median}")
    df['Age'].fillna(age_median, inplace=True)

# Impute embarked with mode
for col in ['embarked', 'embark_town', 'Embarked']:
    if col in df.columns:
        col_mode = df[col].mode()[0]
        print(f"Imputing {col} with mode: {col_mode}")
        df[col].fillna(col_mode, inplace=True)

# Drop redundant columns
redundant_cols = ['embark_town', 'alive', 'alone', 'class', 'who', 'adult_male']
for r_col in redundant_cols:
    if r_col in df.columns:
        print(f"Dropping redundant column: {r_col}")
        df.drop(columns=[r_col], inplace=True)

# Normalize column names if Kaggle dataset was loaded
name_mapping = {
    'Survived': 'survived', 'Pclass': 'pclass', 'Sex': 'sex',
    'Age': 'age', 'SibSp': 'sibsp', 'Parch': 'parch',
    'Fare': 'fare', 'Embarked': 'embarked'
}
df.rename(columns={k: v for k, v in name_mapping.items() if k in df.columns}, inplace=True)

# Select primary variables
keep_cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[[c for c in keep_cols if c in df.columns]]

print("\nMissing values after imputation:")
print(df.isnull().sum())

## Step 4: Outlier Detection and Visualization
We visualize the outliers of `age` and `fare` using boxplots before outlier removal.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(y=df['age'], color='skyblue')
plt.title('Boxplot of Age (Before)')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['fare'], color='salmon')
plt.title('Boxplot of Fare (Before)')

plt.tight_layout()
plt.savefig('plots/outliers_before.png', dpi=150)
plt.show()

## Step 5: Handling Outliers (IQR Method)
Using the Interquartile Range (IQR) method, we clean the outliers from the dataset.

In [ ]:
# Calculate IQR and bounds for age
q1_age = df['age'].quantile(0.25)
q3_age = df['age'].quantile(0.75)
iqr_age = q3_age - q1_age
lower_age = q1_age - 1.5 * iqr_age
upper_age = q3_age + 1.5 * iqr_age
print(f"Age IQR: {iqr_age:.2f}, bounds: [{lower_age:.2f}, {upper_age:.2f}]")

# Calculate IQR and bounds for fare
q1_fare = df['fare'].quantile(0.25)
q3_fare = df['fare'].quantile(0.75)
iqr_fare = q3_fare - q1_fare
lower_fare = q1_fare - 1.5 * iqr_fare
upper_fare = q3_fare + 1.5 * iqr_fare
print(f"Fare IQR: {iqr_fare:.2f}, bounds: [{lower_fare:.2f}, {upper_fare:.2f}]")

# Filter outliers
before_count = len(df)
df_cleaned = df[
    (df['age'] >= lower_age) & (df['age'] <= upper_age) &
    (df['fare'] >= lower_fare) & (df['fare'] <= upper_fare)
].copy()

after_count = len(df_cleaned)
print(f"Removed {before_count - after_count} outlier rows. Remaining rows: {after_count}")

# Plot outliers after removal
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(y=df_cleaned['age'], color='lightblue')
plt.title('Boxplot of Age (After)')

plt.subplot(1, 2, 2)
sns.boxplot(y=df_cleaned['fare'], color='lightcoral')
plt.title('Boxplot of Fare (After)')

plt.tight_layout()
plt.savefig('plots/outliers_after.png', dpi=150)
plt.show()

## Step 6: Categorical Encoding
We convert categorical features (`sex`, `embarked`) into numerical representations using One-Hot Encoding.

In [ ]:
print("Applying One-Hot Encoding to 'sex' and 'embarked'...")
df_encoded = pd.get_dummies(df_cleaned, columns=['sex', 'embarked'], drop_first=True)

# Convert boolean to int
bool_cols = df_encoded.select_dtypes(include=['bool']).columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print("Encoded columns:", df_encoded.columns.tolist())
df_encoded.head()

## Step 7: Feature Scaling
We normalize/standardize the numerical features (`age` and `fare`). We demonstrate standard scaling (mean=0, std=1) and Min-Max scaling (range=[0, 1]).

In [ ]:
# Z-score scaling
df_encoded['age_standardized'] = (df_encoded['age'] - df_encoded['age'].mean()) / df_encoded['age'].std()
df_encoded['fare_standardized'] = (df_encoded['fare'] - df_encoded['fare'].mean()) / df_encoded['fare'].std()

# Min-max scaling
df_encoded['age_minmax'] = (df_encoded['age'] - df_encoded['age'].min()) / (df_encoded['age'].max() - df_encoded['age'].min())
df_encoded['fare_minmax'] = (df_encoded['fare'] - df_encoded['fare'].min()) / (df_encoded['fare'].max() - df_encoded['fare'].min())

print("Scaled columns statistics:")
print(df_encoded[['age', 'age_standardized', 'age_minmax', 'fare', 'fare_standardized', 'fare_minmax']].describe().loc[['mean', 'std', 'min', 'max']])

## Step 8: Exporting Cleaned Dataset
Finally, we save our cleaned and preprocessed dataset to `cleaned_titanic.csv`.

In [ ]:
output_path = 'cleaned_titanic.csv'
df_encoded.to_csv(output_path, index=False)
print(f"Saved cleaned & preprocessed dataset to '{output_path}'. Final shape: {df_encoded.shape}")